# Data Preprocessing for taiko-siffusion

## Unpack .osz Files

Unpack all `.osz` files from `sample_data/raw/*.osz` into `sample_data/unpacked/`.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from src.preprocessing.unpack_osz import unpack_osz_files

source_glob = "sample_data/raw/*.osz"
destination_root = "sample_data/unpacked"

extracted = unpack_osz_files(
    source_glob=source_glob,
    destination_root=destination_root,
    overwrite=False,
)

print(f"Processed {len(extracted)} archive(s).")

Unpacking .osz files: 100%|██████████| 5/5 [00:00<00:00, 12.78file/s]

Processed 5 archive(s).


## Process into intermediate files

Below is a sample that parses one selected beatmap, then reconstruct the beatmap using the same parsed files. The parsed *.notes.json files should be used for training along with the corresponding *.mp3 or *.ogg file.

The map picked in this example is: 2516534 OU - Mountain (FORMless000) [dang dang di dang]

In [ ]:
from pathlib import Path
from src.preprocessing.osutaiko_parser import parse_osu_file_to_jsons

#### EDIT THESE NAMES ####
map_name = "OU - Mountain (FORMless000) [dang dang di dang]"
map_id = "2516534"
##########################

osu_path = Path(f"sample_data/unpacked/{map_id}/{map_name}.osu")
out_dir = Path(f"sample_data/unpacked/{map_id}/parsed")

parse_osu_file_to_jsons(
    osu_path=osu_path,
    out_dir=out_dir,
    include_bpm_events=True,
)

In [4]:
# Inspections

import json

notes_path = out_dir / f"{map_name}.notes.json"
timing_path = out_dir / f"{map_name}.timing.json"
metadata_path = out_dir / f"{map_name}.metadata.json"

with notes_path.open("r", encoding="utf-8") as f:
    notes = json.load(f)

with timing_path.open("r", encoding="utf-8") as f:
    timing = json.load(f)

with metadata_path.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

print("First 10 note events:")
for event in notes["notes"][:10]:
    print(event)

print("\nFirst 5 timing points:")
for tp in timing["timing_points"][:5]:
    print(tp)

print("\nMetadata title:", metadata["metadata"]["Title"])

# Check BPM change events

bpm_events = [n for n in notes["notes"] if n["type"] == "bpmchange"]
print(f"Found {len(bpm_events)} bpmchange events")
bpm_events[:5]

First 10 note events:
{'type': 'bpmchange', 'time': 3385, 'raw_time': 3385, 'sv': 210.0, 'volume': 70, 'bpm': 150.0, 'meter': 4}
{'type': 'don', 'time': 5085, 'raw_time': 5085, 'sv': 207.54810274863414, 'volume': 70, 'bpm': None, 'meter': None}
{'type': 'kat', 'time': 5385.0, 'raw_time': 5385, 'sv': 186.65280467752117, 'volume': 70, 'bpm': None, 'meter': None}
{'type': 'don', 'time': 5485.0, 'raw_time': 5485, 'sv': 198.02346693186638, 'volume': 70, 'bpm': None, 'meter': None}
{'type': 'kat', 'time': 5585.0, 'raw_time': 5585, 'sv': 182.4738597523625, 'volume': 70, 'bpm': None, 'meter': None}
{'type': 'don', 'time': 5785.0, 'raw_time': 5785, 'sv': 191.167948121429, 'volume': 70, 'bpm': None, 'meter': None}
{'type': 'kat', 'time': 5885.0, 'raw_time': 5885, 'sv': 176.38021437123896, 'volume': 70, 'bpm': None, 'meter': None}
{'type': 'don', 'time': 6085.0, 'raw_time': 6085, 'sv': 184.54976551608135, 'volume': 70, 'bpm': None, 'meter': None}
{'type': 'kat', 'time': 6285.0, 'raw_time': 6285, 

[{'type': 'bpmchange',
  'time': 3385,
  'raw_time': 3385,
  'sv': 210.0,
  'volume': 70,
  'bpm': 150.0,
  'meter': 4},
 {'type': 'bpmchange',
  'time': 32185.0,
  'raw_time': 32185,
  'sv': 236.65185138142493,
  'volume': 70,
  'bpm': 150.0,
  'meter': 5},
 {'type': 'bpmchange',
  'time': 33985.0,
  'raw_time': 33985,
  'sv': 252.00000000000009,
  'volume': 70,
  'bpm': 150.0,
  'meter': 4},
 {'type': 'bpmchange',
  'time': 56385.0,
  'raw_time': 56385,
  'sv': 210.0,
  'volume': 70,
  'bpm': 150.0,
  'meter': 4},
 {'type': 'bpmchange',
  'time': 85185.0,
  'raw_time': 85185,
  'sv': 251.04253119488456,
  'volume': 70,
  'bpm': 150.0,
  'meter': 5}]

## Reconstruct beatmaps from parsed files

In [ ]:
from pathlib import Path
from src.preprocessing.osutaiko_reconstructor import reconstruct_osu

# Full metadata test

notes_path = Path(out_dir / f"{map_name}.notes.json")
timing_path = Path(out_dir / f"{map_name}.timing.json")
metadata_path = Path(out_dir / f"{map_name}.metadata.json")
out_path = Path(out_dir / f"{map_name}.reconstructed.osu")

reconstruct_osu(
    notes_path=notes_path,
    timing_path=timing_path,
    metadata_path=metadata_path,
    out_path=out_path,
)

print(out_path.read_text(encoding="utf-8")[:2000])

# Note data only test
# The model will only generate the note sequence; 
# in practice, metadata should be filled by hand, 
# and timing points will be auto-generated.

notes_only_out = Path(out_dir / f"{map_name}.notes_only.reconstructed.osu")

reconstruct_osu(
    notes_path=notes_path,
    timing_path=None,
    metadata_path=None,
    out_path=notes_only_out,
)

print(notes_only_out.read_text(encoding="utf-8")[:2000])

# Check BPM change events

import json

with notes_path.open("r", encoding="utf-8") as f:
    notes_json = json.load(f)

bpm_events = [n for n in notes_json["notes"] if n["type"] == "bpmchange"]
print("BPM events in notes:", len(bpm_events))
print("Example:", bpm_events[:3])

osu file format v14

[General]
AudioFilename:audio.mp3
AudioLeadIn:0
PreviewTime:106185
Countdown:0
SampleSet:None
StackLeniency:0.7
Mode:1
LetterboxInBreaks:0
WidescreenStoryboard:0

[Editor]
DistanceSpacing:1
BeatDivisor:4
GridSize:32
TimelineZoom:1

[Metadata]
Title:Mountain
TitleUnicode:山
Artist:OU
ArtistUnicode:OU
Creator:FORMless000
Version:dang dang di dang
Source:
Tags:
BeatmapID:5550429
BeatmapSetID:2516534

[Difficulty]
HPDrainRate:4
CircleSize:5.5
OverallDifficulty:5.5
ApproachRate:5.5
SliderMultiplier:1.4
SliderTickRate:1.0

[Events]
//Background and Video events
//Break Periods

[TimingPoints]
3385,400,4,1,0,70,1,0
4980,-100,4,1,0,70,0,0
5080,-101.181363365357,4,1,0,70,0,0
5385,-112.508354944259,4,1,0,70,0,0
5480,-106.048037262298,4,1,0,70,0,0
5585,-115.084977259205,4,1,0,70,0,0
5780,-109.851050902429,4,1,0,70,0,0
5885,-119.060973334571,4,1,0,70,0,0
6080,-113.790445310374,4,1,0,70,0,0
6285,-124.576794505794,4,1,0,70,0,0
6380,-117.871111268968,4,1,0,70,0,0
6585,-128.8807172